# Aula 01: Fundamentos de Deep Learning, Neurônio Artificial & Funções de Ativação

Bem-vindo(a) à primeira aula do nosso curso de PyTorch! Nesta aula, vamos desmistificar o funcionamento de uma rede neural artificial focando em sua unidade básica: o **Neurônio Artificial (ou Perceptron)** e nas **Funções de Ativação**.

Vamos contextualizar esse aprendizado com um cenário de negócios do mundo real: classificar a performance de atletas baseando-se no **consumo diário de Creatina** e nas **horas de treino de alta intensidade semanais**.

## Objetivos desta aula:
1. Configurar o ambiente local utilizando o gerenciador moderno **`uv`**.
2. Compreender o fluxo de cálculo de um neurônio artificial (Forward Pass: combinação linear e função de ativação).
3. Explorar as **Funções de Ativação (Sigmoid, ReLU)** e entender graficamente como elas permitem resolver problemas não-lineares.
4. Explorar a intuição geométrica da fronteira de decisão e do viés (*bias*) na planilha interativa.
5. Dominar as estruturas de **Tensores no PyTorch** e suas principais operações (*slicing*, *broadcasting* e redimensionamento).
6. Construir o *forward pass* manual de um Perceptron com álgebra de tensores.

> **Nota Importante**: O ciclo de treinamento automático da rede (usando `nn.Module`, `Autograd`, funções de perda e otimizadores por gradiente descendente) será o foco central da **Aula 2**.

--- 

## 1. Configurando o Ambiente de Desenvolvimento Local com `uv`

Para rodar este notebook e os projetos do curso de forma profissional na sua máquina, siga os passos abaixo para preparar seu ambiente virtual isolado utilizando o **`uv`** (um gerenciador de pacotes ultra-rápido escrito em Rust):

### Passo a Passo de Setup:

1. **Instalar o `uv`** (se ainda não o tiver):
   * *Linux/macOS*:
     ```bash
     curl -LsSf https://astral.sh/uv/install.sh | sh
     ```
   * *Windows*:
     ```powershell
     powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
     ```

2. **Criar o ambiente virtual** na pasta raiz do projeto:
   ```bash
   uv venv
   ```

3. **Ativar o ambiente virtual**:
   * *Linux/macOS*:
     ```bash
     source .venv/bin/activate
     ```
   * *Windows*:
     ```powershell
     .venv\Scripts\activate
     ```

4. **Instalar as dependências** recomendadas para esta aula:
   ```bash
   uv pip install torch torchvision pandas numpy matplotlib ipykernel
   ```

5. **Registrar o kernel no Jupyter** (caso utilize o VS Code ou Jupyter fora do ambiente global):
   ```bash
   python -m ipykernel install --user --name=curso-pytorch --display-name "Python (Curso PyTorch)"
   ```

*Nota: Lembre-se de selecionar o kernel "Python (Curso PyTorch)" no canto superior direito do seu editor de notebook no VS Code.*

--- 

## 2. O Neurônio Artificial (Perceptron) & Dinâmica Teórica

Inspirado no neurônio biológico, o neurônio artificial recebe múltiplas entradas, realiza um cálculo linear ponderado e passa o resultado por uma função de ativação para gerar uma classificação.

Dado um conjunto de entradas $x = [x_1, x_2]$, o neurônio executa duas etapas sequenciais:

### Passo 1: Combinação Linear
Multiplicamos cada entrada $x_i$ por um peso associado $w_i$, que indica a importância daquela entrada para a decisão. Somamos todos esses produtos e adicionamos o viés (**bias**, indicado por $b$):

$$z = w_1 x_1 + w_2 x_2 + b$$

### Passo 2: Função de Ativação
Para limitar a saída a um intervalo interpretável ou introduzir não-linearidade, passamos $z$ por uma função $f(z)$.

### 📊 Desafio Prático com a Planilha Interativa

Antes de rodar o código, vivencie o ajuste de parâmetros manualmente na planilha contida na pasta:
👉 **[simulacao_perceptron.xlsx](simulacao_perceptron.xlsx)**

Modifique os valores de **W1 (Célula M2)**, **W2 (Célula M3)** e **Bias (Célula M4)** para tentar zerar o erro dos 20 atletas e diminuir a **Loss Total (Célula M6)**.

#### 🔍 Questões para Reflexão Teórica:
1. **O Efeito do Bias (Viés):** O que acontece com a fronteira de decisão (reta) e com a Loss quando você tenta ajustar os pesos mantendo o Bias fixado em $0$? Por que o viés é geometricamente essencial?
2. **Cálculo Manual:** Pegue o primeiro atleta da lista na planilha (Ana: Creatina = 5g, Treino = 8h, Classe Real = 1). Calcule a combinação linear $z$ e a saída Sigmoid $\hat{y}$ usando os pesos finais que você encontrou. A predição do neurônio foi correta?

--- 

## 3. Funções de Ativação e a Resolução de Problemas Não-Lineares

Sem funções de ativação, o empilhamento de múltiplas camadas lineares se resume a uma única transformação linear ($W_2 W_1 x = W_{eq} x$). As funções de ativação quebram essa limitação e permitem que as redes neurais aprendam fronteiras de decisão complexas e curvas.

### As Principais Funções:
* **Sigmoid**: $\sigma(z) = \frac{1}{1 + e^{-z}}$ — Converte valores em probabilidades no intervalo $[0, 1]$.
* **ReLU (Rectified Linear Unit)**: $f(z) = \max(0, z)$ — Zera valores negativos e mantém os positivos. É a função mais popular nas camadas ocultas por evitar o desvanecimento do gradiente e ser computacionalmente leve.
* **Leaky ReLU**: $f(z) = \max(\alpha z, z)$ — Permite um pequeno gradiente em valores negativos para evitar 'neurônios mortos'.

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

# Visualizando graficamente as funções de ativação
z = torch.linspace(-5, 5, 200)

sigmoid = torch.sigmoid(z)
relu = torch.relu(z)
leaky_relu = torch.nn.functional.leaky_relu(z, negative_slope=0.1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.plot(z.numpy(), sigmoid.numpy(), color='blue', linewidth=2)
plt.title(r'Sigmoid: $\sigma(z)$')
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 2)
plt.plot(z.numpy(), relu.numpy(), color='green', linewidth=2)
plt.title(r'ReLU: $\max(0, z)$')
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 3)
plt.plot(z.numpy(), leaky_relu.numpy(), color='red', linewidth=2)
plt.title(r'Leaky ReLU: $\max(0.1z, z)$')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 💡 Demonstração Visual: Por que precisamos de não-linearidades?

Observe abaixo como um separador linear (sem ativações não-lineares) falha ao tentar classificar um problema não-linearmente separável (como o padrão XOR), enquanto funções de ativação permitem dobrar o espaço e criar fronteiras não-lineares:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Gerando dados não-lineares (estilo XOR)
np.random.seed(42)
X_xor = np.random.uniform(-2, 2, (200, 2))
Y_xor = (X_xor[:, 0] * X_xor[:, 1] > 0).astype(int)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Modelo Linear tentando separar XOR
ax1.scatter(X_xor[Y_xor==0, 0], X_xor[Y_xor==0, 1], color='red', label='Classe 0')
ax1.scatter(X_xor[Y_xor==1, 0], X_xor[Y_xor==1, 1], color='blue', label='Classe 1')
ax1.plot([-2, 2], [0.4, -0.4], color='black', linestyle='--', linewidth=2, label='Fronteira Linear (Reta)')
ax1.set_title('Sem Ativação Não-Linear\n[Falha]: Impossível separar com uma reta')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Modelo Não-Linear (Rede com Ativações)
ax2.scatter(X_xor[Y_xor==0, 0], X_xor[Y_xor==0, 1], color='red', label='Classe 0')
ax2.scatter(X_xor[Y_xor==1, 0], X_xor[Y_xor==1, 1], color='blue', label='Classe 1')
xx, yy = np.meshgrid(np.linspace(-2.2, 2.2, 200), np.linspace(-2.2, 2.2, 200))
Z_bound = (xx * yy > 0).astype(int)
ax2.contour(xx, yy, Z_bound, levels=[0.5], colors='green', linewidths=2.5)
ax2.plot([], [], color='green', linewidth=2.5, label='Fronteira Não-Linear (Ativada)')
ax2.set_title('Com Ativação Não-Linear (ReLU/Sigmoid)\n[Sucesso]: Curva o espaço e separa as classes')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

--- 

## 4. Bootcamp de Tensores no PyTorch

Tensores são a estrutura de dados principal do PyTorch. Eles são arrays multidimensionais (como matrizes e vetores) otimizados para rodar em GPUs e calcular gradientes automaticamente.

In [ ]:
# 1. Criando tensores de diferentes dimensões (ranks)
escalar = torch.tensor(3.1415)                   # Rank 0 (0D)
vetor = torch.tensor([1.0, 2.0, 3.0])            # Rank 1 (1D) - Shape [3]
matriz = torch.tensor([[1.0, 2.0], [3.0, 4.0]])  # Rank 2 (2D) - Shape [2, 2]

print("Escalar:", escalar, "| Rank:", escalar.ndim)
print("Vetor:", vetor, "| Shape:", vetor.shape)
print("Matriz:\n", matriz, "| Shape:", matriz.shape)

### 4.1 Operações Matemáticas e Álgebra Linear
Operações comuns como soma (`+`) e multiplicação (`*`) ocorrem elemento a elemento (*element-wise*). Já para operações matriciais legítimas, usamos `@` ou `torch.matmul`.

In [ ]:
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
b = torch.tensor([[2.0, 0.0], [1.0, 2.0]])

print("Multiplicação Elemento a Elemento (a * b):\n", a * b)
print("Multiplicação de Matrizes (a @ b):\n", a @ b)

### 📝 Desafios Práticos de Tensores (Complete o Código!)

Escreva o código em Python para resolver cada um dos desafios abaixo utilizando as funções nativas do PyTorch. Isso ajudará a construir fluência com manipulação de dados.

In [ ]:
# DESAFIO 1: Slicing e Indexação
# Crie uma matriz A de tamanho (4, 4) preenchida com números aleatórios (use torch.randn).
# Extraia apenas as duas linhas do meio (linhas de índice 1 e 2) e as duas últimas colunas (índices 2 e 3).

# -- ESCREVA SEU CÓDIGO AQUI --



# DESAFIO 2: Broadcasting
# Crie uma matriz M de shape (3, 3) e um vetor V de shape (3,).
# Some M e V. Explique mentalmente como o PyTorch expandiu o vetor V para realizar a soma.

# -- ESCREVA SEU CÓDIGO AQUI --



# DESAFIO 3: Redimensionamento e Shapes
# Crie um tensor T de shape (6,).
# Transforme T em uma matriz de shape (2, 3) usando o método .view().
# Adicione uma dimensão extra no início para que seu shape final seja (1, 2, 3) usando o método .unsqueeze().

# -- ESCREVA SEU CÓDIGO AQUI --


--- 

## 5. O Dataset de Negócio e a Visualização dos Dados

Vamos recriar o dataset dos atletas em nosso Notebook usando **Pandas** e visualizar a distribuição dos dados com **Matplotlib** para entender a fronteira de decisão linear.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Dados reais/fictícios da planilha dos atletas
dados = {
    "Atleta": ["Ana", "Bruno", "Carlos", "Diana", "Eduardo", "Fernanda", "Gabriel", "Helena", "Igor", "Julia", 
               "Kleber", "Larissa", "Mauricio", "Nara", "Otavio", "Patricia", "Quirino", "Rita", "Samuel", "Tatiana"],
    "Creatina_g": [5, 1, 3, 0, 8, 2, 6, 4, 10, 2, 7, 3, 5, 1, 9, 0, 4, 2, 8, 3],
    "Treino_h": [8, 2, 6, 4, 7, 3, 5, 2, 9, 5, 4, 2, 6, 6, 3, 8, 8, 5, 2, 3],
    "Performance_Real": [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0]
}

df = pd.DataFrame(dados)

# Plotando os dados dos atletas no gráfico de dispersão
plt.figure(figsize=(8, 6))
alta_perf = df[df['Performance_Real'] == 1]
reg_perf = df[df['Performance_Real'] == 0]

plt.scatter(alta_perf['Creatina_g'], alta_perf['Treino_h'], color='green', marker='^', s=100, label='Alta Performance (1)')
plt.scatter(reg_perf['Creatina_g'], reg_perf['Treino_h'], color='red', marker='o', s=100, label='Performance Regular (0)')

plt.xlabel('Consumo de Creatina (g/dia)')
plt.ylabel('Horas de Treino de Alta Intensidade / semana')
plt.title('Distribuição de Performance dos Atletas')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

--- 

## 6. Forward Pass Manual do Perceptron em PyTorch (Sem Treinamento)

Vamos realizar a inferência (*Forward Pass*) manual de um Perceptron com PyTorch, multiplicando as matrizes de entrada pelos pesos e aplicando a função Sigmoid. 

Nesta etapa, estamos usando pesos fixos para entender o fluxo matemático da predição sem o treinamento automático.

In [ ]:
import torch

# Convertendo colunas do DataFrame para tensores do PyTorch
X = torch.tensor(df[['Creatina_g', 'Treino_h']].values, dtype=torch.float32)  # Shape: [20, 2]
Y_real = torch.tensor(df['Performance_Real'].values, dtype=torch.float32).unsqueeze(1)  # Shape: [20, 1]

# Definindo pesos (W1, W2) e bias (b) manualmente
W = torch.tensor([[1.5], [1.0]], dtype=torch.float32)  # Shape: [2, 1]
b = torch.tensor([-8.5], dtype=torch.float32)            # Shape: [1]

# 1. Combinação linear z = X @ W + b
z = X @ W + b

# 2. Aplicação da Ativação Sigmoid para obter probabilidades
predicoes_prob = torch.sigmoid(z)

# 3. Classificação com limite em 0.5
classes_preditas = (predicoes_prob >= 0.5).float()

# Calculando a acurácia manual
acuracia = (classes_preditas == Y_real).float().mean() * 100
print(f"Acurácia do Forward Pass Manual: {acuracia.item():.2f}%")

--- 

## 7. Conclusão da Aula 01 & Próximos Passos

Parabéns por concluir a primeira aula! 

### O que aprendemos hoje:
- Como estruturar o ambiente com `uv`.
- A intuição teórica e geométrica do neurônio artificial, dos pesos e do viés (*bias*).
- As principais funções de ativação (**Sigmoid, ReLU**) e o papel da não-linearidade.
- Como operar Tensores no PyTorch (`shape`, `broadcasting`, `matmul`).
- Como executar o *Forward Pass* manual usando PyTorch.

👉 **Na Aula 02**: Vamos construir o ciclo completo de treinamento automático de uma Rede Neural! Veremos o motor de diferenciação automática **Autograd**, declararemos modelos com `nn.Module`, e utilizaremos otimizadores (como `SGD` e `Adam`) para aprender os pesos a partir dos dados em uma MLP aplicada à previsão de Churn.